In [ ]:
import os
import glob
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm

# --- 1. 파일 목록 확보 ---
folders = ["C:/workspace/carsim_processed_data_25hz"]
all_files = []
for f in folders: 
    all_files.extend(glob.glob(os.path.join(f, "*.csv")))

seq_len = 4
stride = 1

# 🌟 원하는 Horizon 스텝 설정
num_steps = 10
lookahead_margin = num_steps + 1  

# --- 2. 통계량(mean, std) 계산 ---
all_samples, all_accels = [], []

print("슬라이딩 윈도우 기반 글로벌 표준화 파라미터 계산 중...")
for f in tqdm(all_files, desc="통계량 산출 CSV 분석"):
    df = pd.read_csv(f)
    
    # 🌟 제어 입력 변화량 계산 (Brake 제외)
    df['d_Throttle'] = df['Throttle'].diff().fillna(0.0)
    df['d_delta'] = df['delta'].diff().fillna(0.0)

    states = df[['vx', 'vy', 'dpsi']].values
    ctrls = df[['Throttle', 'delta']].values  # 🌟 Brake 제거
    d_ctrls = df[['d_Throttle', 'd_delta']].values  # 🌟 d_Brake 제거
    accels = df[['ax', 'ay']].values
    
    # 신경망의 입력 피처: [상태(3), 제어입력(2), 제어변화량(2)] -> 총 7차원
    combined_features = np.hstack((states, ctrls, d_ctrls))

    for start_idx in range(0, len(states) - seq_len - lookahead_margin, stride):
        sample = combined_features[start_idx : start_idx + seq_len]
        all_samples.append(sample)
        all_accels.append(accels[start_idx : start_idx + seq_len])
        
concat_inp = np.vstack(all_samples)
concat_acc = np.vstack(all_accels)

global_stats = {
    'in_mean': concat_inp.mean(axis=0), 
    'in_std': concat_inp.std(axis=0),
    'accel_std': concat_acc.std(axis=0)
}
print("글로벌 데이터 통계량 계산 완료.")

# --- 3. 로컬 텐서 리스트 채우기 ---
in_mean_t = torch.tensor(global_stats['in_mean'], dtype=torch.float32)
in_std_t = torch.tensor(global_stats['in_std'], dtype=torch.float32)

all_processed_inputs, all_curr_raw, all_ctrl, all_d_ctrl, all_accel = [], [], [], [], []

target_containers = {f't{i}': [] for i in range(1, num_steps + 1)}          
ctrl_containers = {f'ctrl_t{i}': [] for i in range(1, num_steps + 1)}          
d_ctrl_containers = {f'd_ctrl_t{i}': [] for i in range(1, num_steps + 1)}          

for f in tqdm(all_files, desc="최종 데이터셋 텐서 빌드 및 정규화"):
    df = pd.read_csv(f)
    
    df['d_Throttle'] = df['Throttle'].diff().fillna(0.0)
    df['d_delta'] = df['delta'].diff().fillna(0.0)

    states = torch.tensor(df[['vx', 'vy', 'dpsi']].values, dtype=torch.float32)
    ctrls = torch.tensor(df[['Throttle', 'delta']].values, dtype=torch.float32) # 🌟 Brake 제거
    d_ctrls = torch.tensor(df[['d_Throttle', 'd_delta']].values, dtype=torch.float32) # 🌟 d_Brake 제거
    accels = torch.tensor(df[['ax', 'ay']].values, dtype=torch.float32)

    for start_idx in range(0, len(states) - seq_len - lookahead_margin, stride):
        curr_idx = start_idx + seq_len - 1
        
        # 🌟 입력 시퀀스 묶기: [seq_len, 7] 차원
        in_unnorm = torch.cat([
            states[start_idx : start_idx + seq_len], 
            ctrls[start_idx : start_idx + seq_len],
            d_ctrls[start_idx : start_idx + seq_len]
        ], dim=1)
        
        all_processed_inputs.append((in_unnorm - in_mean_t) / (in_std_t + 1e-6))
        
        all_curr_raw.append(states[curr_idx])
        all_ctrl.append(ctrls[curr_idx])
        all_d_ctrl.append(d_ctrls[curr_idx])
        all_accel.append(accels[curr_idx])
        
        for i in range(1, num_steps + 1):
            target_containers[f't{i}'].append(states[curr_idx + i])
            ctrl_containers[f'ctrl_t{i}'].append(ctrls[curr_idx + i])
            d_ctrl_containers[f'd_ctrl_t{i}'].append(d_ctrls[curr_idx + i])

# --- 4. 딕셔너리로 저장 ---
dataset_dict = {
    'processed_inputs': torch.stack(all_processed_inputs),
    'curr_raw': torch.stack(all_curr_raw),
    'ctrl': torch.stack(all_ctrl),
    'd_ctrl': torch.stack(all_d_ctrl),
    'accel': torch.stack(all_accel),
    'global_stats': global_stats          
}

for k, v in target_containers.items(): dataset_dict[k] = torch.stack(v)
for k, v in ctrl_containers.items(): dataset_dict[k] = torch.stack(v)
for k, v in d_ctrl_containers.items(): dataset_dict[k] = torch.stack(v)

torch.save(dataset_dict, 'processed_dataset_25hz_carsim_base.pt')
print(f"🎉 데이터셋 빌드 완료! Brake 정보 제거 및 연산 최적화 (Input Dim: 7)")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
import scipy.io as sio
from tqdm import tqdm
import numpy as np

# --- 설정값 ---
max_num_steps = 10
dt = 0.04
m = 1270
lf = 1.015
lr = 1.895

# --- 1. Dataset ---
class PreProcessedVehicleDataset(Dataset):
    def __init__(self, data_dict):
        self.processed_inputs = data_dict['processed_inputs'].float()
        self.curr_raw = data_dict['curr_raw'].float()
        self.ctrl = data_dict['ctrl'].float()
        self.accel = data_dict['accel'].float()
        
        targets_list = [data_dict[f't{i}'].float() for i in range(1, max_num_steps + 1)]
        self.future_targets = torch.stack(targets_list, dim=1)

    def __len__(self):
        return len(self.processed_inputs)

    def __getitem__(self, idx):
        return (self.processed_inputs[idx], self.curr_raw[idx], self.ctrl[idx], 
                self.accel[idx], self.future_targets[idx])

# --- 2. 모델 ---
class ResidualBlock(nn.Module):
    def __init__(self, dim, p=0.1):
        super().__init__()
        self.block = nn.Sequential(nn.Linear(dim, dim), nn.Tanh(), nn.Dropout(p))
    def forward(self, x): return self.block(x)

class DeepDynamicsModel(nn.Module):
    def __init__(self, input_dim=7, hidden_dim=64, p=0.1):
        super(DeepDynamicsModel, self).__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, batch_first=True)
        layers = [nn.Linear(hidden_dim, hidden_dim), nn.Tanh(), nn.Dropout(p)]
        for _ in range(1): layers.append(ResidualBlock(hidden_dim, p))
        self.mlp_backbone = nn.Sequential(*layers)
        self.output_layer = nn.Linear(hidden_dim, 17)
        self.sigmoid = nn.Sigmoid()

        self.register_buffer('t_min', torch.tensor([1, 0.5, 500, -3.0, -0.1, -500.0]))
        self.register_buffer('t_max', torch.tensor([30, 3, 40000.0, 0, 0.1, 500.0]))
        self.register_buffer('dr_min', torch.tensor([500, 0.0, 150.0, 0.2]))
        self.register_buffer('dr_max', torch.tensor([40000.0, 1, 1700.0, 1.7]))
        self.register_buffer('Iz_min', torch.tensor([500.0]))
        self.register_buffer('Iz_max', torch.tensor([2000.0]))

    def forward(self, input_seq):
        _, h_n = self.gru(input_seq)
        x = self.mlp_backbone(h_n[-1])
        out = self.sigmoid(self.output_layer(x))
        tf = self.t_min + (self.t_max - self.t_min) * out[:, 0:6]
        tr = self.t_min + (self.t_max - self.t_min) * out[:, 6:12]
        dr = self.dr_min + (self.dr_max - self.dr_min) * out[:, 12:16]
        Iz = self.Iz_min + (self.Iz_max - self.Iz_min) * out[:, 16:17]
        return tf, tr, dr, Iz

# --- 3. Physics Loss 계산 함수 ---
def velocity_step(state, ctrl, tf, tr, dr, iz):
    vx, vy, r = state[:, 0], state[:, 1], state[:, 2]
    T, delta = ctrl[:, 0], ctrl[:, 1]
    
    Frx = (dr[:, 0] - dr[:, 1] * vx) * T - dr[:, 2] - dr[:, 3] * (vx**2)
    af = delta - torch.atan2(r * lf + vy, vx + 1e-6) + tf[:, 4]
    ar = torch.atan2(r * lr - vy, vx + 1e-6) + tr[:, 4]
    Ffy = tf[:, 5] + tf[:, 2] * torch.sin(tf[:, 1] * torch.atan(tf[:, 0]*af - tf[:, 3]*(tf[:, 0]*af - torch.atan(tf[:, 0]*af))))
    Fry = tr[:, 5] + tr[:, 2] * torch.sin(tr[:, 1] * torch.atan(tr[:, 0]*ar - tr[:, 3]*(tr[:, 0]*ar - torch.atan(tr[:, 0]*ar))))
    
    vx_dot = (- Ffy * torch.sin(delta) + Frx) / m + vy * r
    vy_dot = (Ffy * torch.cos(delta) + Fry) / m - vx * r
    r_dot = (Ffy * torch.cos(delta) * lf - Fry * lr) / (iz.squeeze() + 1e-6)
    return torch.stack([vx + vx_dot * dt, vy + vy_dot * dt, r + r_dot * dt], dim=1)

# --- 4. 메인 학습 루프 ---
def main_train():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    data_dict = torch.load('processed_dataset_25hz_carsim_base.pt', map_location='cpu', weights_only=False)
    global_stats = data_dict['global_stats']
    
    dataset = PreProcessedVehicleDataset(data_dict)
    train_size = int(0.8 * len(dataset))
    train_ds, val_ds = Subset(dataset, range(train_size)), Subset(dataset, range(train_size, len(dataset)))
    train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=256)
    
    model = DeepDynamicsModel(input_dim=7).to(device)
    optimizer = optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5)
    
    s_std = torch.tensor(global_stats['in_std'][0:3], device=device) + 1e-6
    best_loss = float('inf')

    for epoch in range(110):
        model.train()
        t_loss = 0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            i_s, c_r, ct_r, _, targets = [b.to(device) for b in batch]
            tf, tr, dr, iz = model(i_s)
            st_pred = velocity_step(c_r, ct_r, tf, tr, dr, iz)
            loss = torch.mean(((st_pred - targets[:, 0]))**2)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            t_loss += loss.item()

        # Validation
        model.eval()
        v_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                i_s, c_r, ct_r, _, targets = [b.to(device) for b in batch]
                tf, tr, dr, iz = model(i_s)
                st_pred = velocity_step(c_r, ct_r, tf, tr, dr, iz)
                v_loss += torch.mean(((st_pred - targets[:, 0]))**2).item()
        
        avg_v = v_loss / len(val_loader)
        scheduler.step(avg_v)
        print(f"Epoch {epoch+1} | Train Loss: {t_loss/len(train_loader):.6f} | Val Loss: {avg_v:.6f}")

        if avg_v < best_loss:
            best_loss = avg_v
            torch.save(model.state_dict(), 'best_model.pth')
            weights = {k.replace('.', '_'): v.detach().cpu().numpy() for k, v in model.state_dict().items()}
            weights.update({'in_mean': global_stats['in_mean'], 'in_std': global_stats['in_std']})
            sio.savemat('best_weights_baseline.mat', weights)
            print("🌟 Best Model Saved.")

if __name__ == "__main__":
    main_train()